## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [27]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
# import sendgrid
import os
import smtplib
from email.mime.text import MIMEText
from typing import Dict
from email.mime.multipart import MIMEMultipart

# from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [28]:
load_dotenv(override=True)

True

In [29]:
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENROUTER_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.environ.get("OPENROUTER_BASE_URL")


## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [30]:
# INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
# produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
# words. Capture the main points. Write succintly, no need to have complete sentences or good \
# grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
# essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

# search_agent = Agent(
#     name="Search agent",
#     instructions=INSTRUCTIONS,
#     tools=[WebSearchTool(search_context_size="low")],
#     model="gpt-4o-mini",
#     model_settings=ModelSettings(tool_choice="required"),
# )

INSTRUCTIONS = """You are a research assistant. Given a search term, you search the web
for that term and produce a concise summary of the results.
2–3 paragraphs, <300 words, capture main points only.
"""

@function_tool
def web_search(query: str) -> str:
    """Search the web for information about a topic."""
    # temporary mock (you can later connect SerpAPI, Tavily, etc.)
    return f"Search results for: {query}"

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[web_search],   # ✅ valid function tool
    model="gpt-4o-mini",
)

In [31]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

# tools = [
#     {
#         "type": "function",
#         "function": {
#             "name": "web_search",
#             "description": "Search the web for information",
#             "parameters": {
#                 "type": "object",
#                 "properties": {
#                     "query": {
#                         "type": "string",
#                         "description": "Search query"
#                     }
#                 },
#                 "required": ["query"]
#             }
#         }
#     }
# ]

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


As of 2025, several AI agent frameworks are emerging and solidifying their roles in the development of autonomous systems. Notably, frameworks like OpenAI's ChatGPT and Google's DeepMind are among the front-runners, focusing on natural language processing (NLP) and reinforcement learning for complex decision-making tasks. These frameworks are being optimized for seamless integration with environments requiring adaptability and interaction, catering to varied applications from customer support to autonomous vehicles.

Additionally, the rise of federated learning frameworks like TensorFlow Federated is noteworthy. This approach allows AI agents to learn from decentralized data sources while maintaining privacy, which is increasingly critical in a data-sensitive world. Newer architectures are also being explored, such as those incorporating multi-agent systems for collaboration among AI entities, which enhance problem-solving capabilities in complex environments. Furthermore, the integration of ethical AI practices is becoming paramount, leading to frameworks that emphasize transparency and accountability in AI decision-making processes.

Overall, 2025 is marked by a focus on collaborative, ethical, and adaptable AI frameworks that promise enhanced performance in real-world applications. The advancements in agent-based frameworks suggest a future where AI agents can operate more autonomously while being socially responsible and efficient.

OPENAI_API_KEY is not set, skipping trace export


### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

In [32]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [33]:

message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


searches=[WebSearchItem(reason='To find current information about AI agent frameworks projected in 2025, looking for upcoming trends and predictions in the field.', query='future AI agent frameworks 2025'), WebSearchItem(reason='This search will help gather insights about specific AI frameworks that are expected to be influential in 2025, including their features and applications.', query='AI agent frameworks 2025'), WebSearchItem(reason='To identify companies, products, and research initiatives that are currently developing or are expected to develop AI agents leading into 2025.', query='latest developments in AI agent technology 2025')]


OPENAI_API_KEY is not set, skipping trace export


In [42]:
@function_tool
# def send_email(subject: str, html_body: str) -> Dict[str, str]:
#     """ Send out an email with the given subject and HTML body """
#     sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
#     from_email = Email("ed@edwarddonner.com") # Change this to your verified email
#     to_email = To("ed.donner@gmail.com") # Change this to your email
#     content = Content("text/html", html_body)
#     mail = Mail(from_email, to_email, subject, content).get()
#     sg.client.mail.send.post(request_body=mail)
#     return "success"


def send_email(body: str) -> Dict[str, str]:
    """
    Send out an email with HTML content using Gmail SMTP
    """

    to_email = "sirphil.bizz@gmail.com"

    try:
        msg = MIMEMultipart()
        msg["From"] = os.environ["EMAIL"]
        msg["To"] = to_email
        msg["Subject"] = "Sales Email"

        # Send as HTML
        msg.attach(MIMEText(body, "html"))

        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(os.environ["EMAIL"], os.environ["APP_PASSWORD"])

        server.send_message(msg)
        server.quit()

        return {"status": "success", "message": f"Email sent to {to_email}"}

    except Exception as e:
        return {"status": "error", "message": str(e)}

In [43]:
send_email

FunctionTool(name='send_email', description='Send out an email with HTML content using Gmail SMTP', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x111129790>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [44]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)



In [45]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [46]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

In [47]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Showtime!

In [48]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")




Starting research...
Planning searches...
Will perform 3 searches
Searching...


OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


Finished searching
Thinking about report...


OPENAI_API_KEY is not set, skipping trace export


Finished writing report
Writing email...


OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


Email sent
Hooray!


OPENAI_API_KEY is not set, skipping trace export


### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>